# Regresión logística en 2D con datos linealmente separables

Este cuaderno extiende el ejemplo de regresión logística en 1D.

Ahora cada muestra tiene **dos características**:

$$
\mathbf{x} = [x_1, x_2]
$$

## 1. Datos de entrenamiento

Cada fila representa una muestra.

Usaremos dos grupos claramente separados:

- clase `0`: puntos ubicados hacia la parte inferior izquierda;
- clase `1`: puntos ubicados hacia la parte superior derecha.

Esto permite construir un ejemplo visualmente sencillo de clasificación binaria.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, log_loss

# Cada muestra tiene dos características: x1 y x2
X = np.array([
    [1.0, 1.0],
    [1.2, 1.8],
    [2.0, 1.1],
    [2.2, 2.0],
    [1.6, 2.3],
    [2.4, 1.7],

    [4.0, 4.1],
    [4.3, 5.0],
    [5.1, 4.2],
    [5.3, 5.4],
    [4.7, 4.8],
    [5.5, 4.6],
])

# Etiquetas: las primeras muestras son clase 0 y las restantes clase 1
y = np.array([0, 0, 0, 0, 0, 0,
              1, 1, 1, 1, 1, 1])

datos = pd.DataFrame({
    "x1": X[:, 0],
    "x2": X[:, 1],
    "y": y
})

datos

## 2. Visualización de los datos

Ahora no ubicamos las muestras sobre una recta, sino sobre un plano.

El eje horizontal representa $x_1$ y el eje vertical representa $x_2$.

Si los puntos de una clase quedan agrupados en una zona y los de la otra clase en otra zona, el problema es fácil de separar con una línea.

In [ ]:
plt.figure(figsize=(6, 6))

plt.scatter(X[y == 0, 0], X[y == 0, 1],
            s=120, label="Clase 0", edgecolor="k")

plt.scatter(X[y == 1, 0], X[y == 1, 1],
            s=120, label="Clase 1", edgecolor="k")

plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.title("Datos de entrenamiento en dos dimensiones")
plt.xlim(0, 6.5)
plt.ylim(0, 6.5)
plt.grid(True)
plt.legend()
plt.show()

## 3. Modelo de regresión logística en 2D

En 1D se tenía:

$$
z = b + wx
$$

En 2D la idea es la misma, pero ahora hay dos pesos:

$$
z = b + w_1x_1 + w_2x_2
$$

Luego se aplica la función sigmoide:

$$
P(y=1 \mid x_1,x_2) = \sigma(z)
$$

donde:

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

La salida del modelo sigue siendo una probabilidad.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = np.linspace(-8, 8, 300)
sigma_z = sigmoid(z)

plt.figure(figsize=(8, 4))
plt.plot(z, sigma_z)
plt.axhline(0.5, linestyle="--", linewidth=1, label="Umbral 0.5")
plt.axvline(0, linestyle="--", linewidth=1)
plt.xlabel("z")
plt.ylabel(r"$\sigma(z)$")
plt.title("Función sigmoide")
plt.grid(True)
plt.legend()
plt.show()

## 4. Entrenamiento del modelo

El entrenamiento con `scikit-learn` mantiene el mismo flujo del ejemplo en 1D:

1. crear el modelo,
2. entrenarlo con `.fit(X, y)`,
3. calcular probabilidades con `.predict_proba(X)`,
4. calcular clases con `.predict(X)`.

Se usa `C=10` para reducir la regularización y permitir una frontera de decisión más marcada.

In [ ]:
modelo = LogisticRegression(solver="lbfgs", C=10.0, fit_intercept=True)

modelo.fit(X, y)

b = modelo.intercept_[0]
w1, w2 = modelo.coef_[0]

print(f"Intercepto b = {b:.6f}")
print(f"Peso w1      = {w1:.6f}")
print(f"Peso w2      = {w2:.6f}")
print(f"Clases del modelo = {modelo.classes_}")

## 5. Interpretación de los parámetros

El modelo aprendido tiene esta forma:

$$
z = b + w_1x_1 + w_2x_2
$$

En este ejemplo, los pesos suelen ser positivos.

Eso significa que, al aumentar $x_1$ o $x_2$, aumenta también el valor de $z$.

Como la sigmoide crece cuando $z$ crece, entonces aumenta la probabilidad de pertenecer a la clase `1`.

In [ ]:
z_manual = b + w1 * X[:, 0] + w2 * X[:, 1]
prob_manual = sigmoid(z_manual)

prob_sklearn = modelo.predict_proba(X)

indice_clase_1 = np.where(modelo.classes_ == 1)[0][0]
prob_clase_1 = prob_sklearn[:, indice_clase_1]

comparacion = pd.DataFrame({
    "x1": X[:, 0],
    "x2": X[:, 1],
    "y_real": y,
    "z = b + w1*x1 + w2*x2": z_manual,
    "sigmoid(z)": prob_manual,
    "predict_proba clase 1": prob_clase_1
})

comparacion

## 6. Frontera de decisión en 2D

La frontera de decisión aparece cuando el modelo está justo en el umbral:

$$
P(y=1 \mid x_1,x_2)=0.5
$$

La sigmoide vale $0.5$ cuando:

$$
z=0
$$

Por tanto:

$$
b + w_1x_1 + w_2x_2 = 0
$$

Esa ecuación representa una recta.

Si se despeja $x_2$, queda:

$$
x_2 = -\frac{b + w_1x_1}{w_2}
$$

In [ ]:
# Puntos del eje x1 para dibujar la recta frontera
x1_linea = np.linspace(0, 6.5, 200)

# Ecuación de la frontera: x2 = -(b + w1*x1)/w2
x2_linea = -(b + w1 * x1_linea) / w2

plt.figure(figsize=(6, 6))

plt.scatter(X[y == 0, 0], X[y == 0, 1],
            s=120, label="Clase 0", edgecolor="k")

plt.scatter(X[y == 1, 0], X[y == 1, 1],
            s=120, label="Clase 1", edgecolor="k")

plt.plot(x1_linea, x2_linea, linestyle="--", linewidth=2,
         label="Frontera de decisión")

plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.title("Frontera de decisión en 2D")
plt.xlim(0, 6.5)
plt.ylim(0, 6.5)
plt.grid(True)
plt.legend()
plt.show()

## 7. Regiones de decisión

La recta divide el plano en dos zonas:

- un lado se clasifica como clase `0`;
- el otro lado se clasifica como clase `1`.

Para visualizarlo, se evalúa el modelo sobre muchos puntos del plano.

In [ ]:
# Malla de puntos para evaluar el modelo sobre todo el plano
x1_min, x1_max = 0, 6.5
x2_min, x2_max = 0, 6.5

xx1, xx2 = np.meshgrid(
    np.linspace(x1_min, x1_max, 300),
    np.linspace(x2_min, x2_max, 300)
)

malla = np.c_[xx1.ravel(), xx2.ravel()]

# Predicción de clase sobre cada punto de la malla
pred_malla = modelo.predict(malla).reshape(xx1.shape)

plt.figure(figsize=(6, 6))

plt.contourf(xx1, xx2, pred_malla, alpha=0.25)
plt.contour(xx1, xx2, pred_malla, levels=[0.5], linewidths=2)

plt.scatter(X[y == 0, 0], X[y == 0, 1],
            s=120, label="Clase 0", edgecolor="k")

plt.scatter(X[y == 1, 0], X[y == 1, 1],
            s=120, label="Clase 1", edgecolor="k")

plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.title("Regiones de decisión")
plt.xlim(x1_min, x1_max)
plt.ylim(x2_min, x2_max)
plt.grid(True)
plt.legend()
plt.show()

## 8. Mapa de probabilidad

La frontera de decisión no muestra toda la información.

La regresión logística también permite observar qué tan segura es la predicción.

Cerca de la frontera, la probabilidad está cerca de `0.5`.

Lejos de la frontera, la probabilidad se acerca a `0` o a `1`.

In [ ]:
prob_malla = modelo.predict_proba(malla)[:, indice_clase_1].reshape(xx1.shape)

plt.figure(figsize=(6, 6))

plt.contourf(xx1, xx2, prob_malla, levels=30, alpha=0.75)
plt.colorbar(label=r"$P(y=1 \mid x_1,x_2)$")

plt.contour(xx1, xx2, prob_malla, levels=[0.5],
            linewidths=2, linestyles="--")

plt.scatter(X[y == 0, 0], X[y == 0, 1],
            s=120, label="Clase 0", edgecolor="k")

plt.scatter(X[y == 1, 0], X[y == 1, 1],
            s=120, label="Clase 1", edgecolor="k")

plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.title("Mapa de probabilidad de la clase 1")
plt.xlim(x1_min, x1_max)
plt.ylim(x2_min, x2_max)
plt.grid(True)
plt.legend()
plt.show()

## 9. Predicción de clases

Para convertir probabilidades en clases se usa el umbral `0.5`.

La regla es:

$$
\hat{y} =
\begin{cases}
1, & \text{si } P(y=1 \mid x_1,x_2) \geq 0.5\\
0, & \text{si } P(y=1 \mid x_1,x_2) < 0.5
\end{cases}
$$

Como los datos son separables, el modelo debería clasificar correctamente todas las muestras de entrenamiento.

In [ ]:
predicciones = modelo.predict(X)
exactitud = accuracy_score(y, predicciones)

resultados = pd.DataFrame({
    "x1": X[:, 0],
    "x2": X[:, 1],
    "y_real": y,
    "probabilidad_clase_1": prob_clase_1,
    "y_predicha": predicciones
})

print(f"Exactitud del modelo: {100 * exactitud:.2f}%")
resultados

## 10. Función de costo

La regresión logística se entrena minimizando la entropía cruzada.

Para una muestra:

$$
\ell(y,h) = -\left[y\log(h)+(1-y)\log(1-h)\right]
$$

Para varias muestras:

$$
J = -\frac{1}{m}\sum_{i=1}^{m}
\left[
y^{(i)}\log(h^{(i)})+
(1-y^{(i)})\log(1-h^{(i)})
\right]
$$

En `scikit-learn`, esta optimización se realiza internamente.

In [ ]:
loss = log_loss(y, prob_sklearn, labels=modelo.classes_)

print(f"Log loss sobre entrenamiento: {loss:.6f}")

## 11. Matriz de confusión

La matriz de confusión permite contar aciertos y errores.

En este ejemplo, como las clases están claramente separadas, se espera una diagonal perfecta.

In [ ]:
cm = confusion_matrix(y, predicciones, labels=modelo.classes_)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=modelo.classes_)
disp.plot()
plt.title("Matriz de confusión")
plt.show()

## 12. Efecto de la regularización

El parámetro `C` controla la regularización de forma inversa:

- `C` pequeño: regularización fuerte;
- `C` grande: regularización débil.

Con datos separables, al aumentar `C`, la frontera suele volverse más marcada porque el modelo puede usar pesos de mayor magnitud.

In [ ]:
C_values = [0.1, 1.0, 10.0, 100.0]
resumen_C = []

plt.figure(figsize=(7, 7))

# Datos
plt.scatter(X[y == 0, 0], X[y == 0, 1],
            s=120, label="Clase 0", edgecolor="k")
plt.scatter(X[y == 1, 0], X[y == 1, 1],
            s=120, label="Clase 1", edgecolor="k")

for C in C_values:
    modelo_C = LogisticRegression(solver="lbfgs", C=C, fit_intercept=True)
    modelo_C.fit(X, y)

    b_C = modelo_C.intercept_[0]
    w1_C, w2_C = modelo_C.coef_[0]

    x2_C = -(b_C + w1_C * x1_linea) / w2_C

    resumen_C.append({
        "C": C,
        "b": b_C,
        "w1": w1_C,
        "w2": w2_C,
        "norma_pesos": np.sqrt(w1_C**2 + w2_C**2)
    })

    plt.plot(x1_linea, x2_C, linestyle="--", label=f"C={C}")

plt.xlabel(r"$x_1$")
plt.ylabel(r"$x_2$")
plt.title("Efecto de C sobre la frontera de decisión")
plt.xlim(0, 6.5)
plt.ylim(0, 6.5)
plt.grid(True)
plt.legend()
plt.show()

pd.DataFrame(resumen_C)

## Ejercicios propuestos

1. Cambie la posición de los puntos de la clase `0` y observe cómo cambia la frontera.
2. Cambie la posición de los puntos de la clase `1` y vuelva a entrenar.
3. Agregue un punto de clase `0` dentro de la zona de clase `1`.
4. Agregue un punto de clase `1` dentro de la zona de clase `0`.
5. Compare la frontera obtenida con `C=0.1`, `C=1`, `C=10` y `C=100`.
6. Cambie el umbral de decisión de `0.5` a `0.7` y observe cómo cambian las regiones.